# 🎙️ ELEVEN AUDIO — Estúdio de Voz Profissional (Google Colab GPU)
### Clonagem de Voz, Síntese Neural, Transcrição, Tradução e Dublagem
- **Base:** [VoiceStudio (AGPL-3.0)](https://github.com/debpalash/VoiceStudio) + [Eleven Audio (MIT)](https://github.com/gabrielkendy/eleven-audio)
- **Persistência Total:** Conecta ao seu Google Drive (`/content/drive/MyDrive/ElevenAudio`) para **nunca perder modelos ou áudios gerados**.
- **Acesso Externo:** Túnel Cloudflare público e seguro sem necessidade de cadastro ou tokens.

### 1. Verificação de Aceleração por GPU

In [ ]:
# Verifica se o runtime está configurado com GPU (T4 recomendada)
import subprocess
try:
    gpu_info = subprocess.check_output(['nvidia-smi'], text=True)
    print(gpu_info)
    print("\n✅ GPU detectada com sucesso! Pronto para geração ultra-rápida.")
except Exception as e:
    print("⚠️ ATENÇÃO: Nenhuma GPU NVIDIA detectada!")
    print("Vá no menu superior: 'Ambiente de execução' -> 'Alterar tipo de ambiente de execução' -> Selecione 'T4 GPU' e salve.")

### 2. Montar Google Drive & Configurar Pastas de Persistência

In [ ]:
from google.colab import drive
import os

# 1. Monta o Drive do usuário
drive.mount('/content/drive')

# 2. Cria estrutura permanente no Drive
DRIVE_BASE = '/content/drive/MyDrive/ElevenAudio'
CACHE_HF = os.path.join(DRIVE_BASE, 'cache_hf')
CACHE_TORCH = os.path.join(DRIVE_BASE, 'cache_torch')
PASTA_SAIDAS = os.path.join(DRIVE_BASE, 'saidas')
PASTA_DADOS = os.path.join(DRIVE_BASE, 'dados')

for p in [CACHE_HF, CACHE_TORCH, PASTA_SAIDAS, PASTA_DADOS]:
    os.makedirs(p, exist_ok=True)

# 3. Variáveis de ambiente para reaproveitamento de cache dos modelos
os.environ['HF_HOME'] = CACHE_HF
os.environ['TRANSFORMERS_CACHE'] = CACHE_HF
os.environ['TORCH_HOME'] = CACHE_TORCH
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

print("✅ Google Drive conectado!")
print(f"📁 Seus modelos e áudios ficam salvos em: {DRIVE_BASE}")

### 3. Instalação de Dependências & Download do Eleven Audio

In [ ]:
# Instalação de ffmpeg, uv e pacotes principais
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q uv fastapi uvicorn httpx python-multipart pydub

%cd /content

# Baixa o pacote oficial do Eleven Audio (Gabriel Kendy)
import urllib.request, zipfile

zip_url = 'https://github.com/gabrielkendy/eleven-audio/releases/latest/download/eleven-audio.zip'
zip_path = '/content/eleven-audio.zip'
if not os.path.exists('/content/eleven-audio'):
    print("Baixando pacote Eleven Audio...")
    urllib.request.urlretrieve(zip_url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content')
    print("✅ Eleven Audio descompactado com sucesso!")

# Clona a base do VoiceStudio (motor neural)
if not os.path.exists('/content/VoiceStudio'):
    print("Clonando motor VoiceStudio...")
    !git clone --depth 1 https://github.com/debpalash/VoiceStudio.git /content/VoiceStudio
    print("✅ VoiceStudio clonado!")

# Instala as dependências do motor
%cd /content/VoiceStudio
!pip install -q -r requirements.txt || pip install -q TTS soundfile librosa
%cd /content/eleven-audio
!pip install -q -r requirements.txt
print("\n🚀 Todas as dependências foram instaladas com sucesso!")

### 4. Inicialização do Servidor & Túnel Cloudflare

In [ ]:
import subprocess, time, re, os

# Baixa o binário do cloudflared
if not os.path.exists('/usr/local/bin/cloudflared'):
    print("Baixando Cloudflare Tunnel...")
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

# Inicia o backend do VoiceStudio (porta 3900) em segundo plano
print("Inicializando motor VoiceStudio (Porta 3900)...")
p_base = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'backend.main:app', '--host', '127.0.0.1', '--port', '3900'],
    cwd='/content/VoiceStudio',
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Configura o Eleven Audio para salvar saídas no Google Drive
os.environ['ELEVEN_SAIDAS'] = '/content/drive/MyDrive/ElevenAudio/saidas'
os.environ['ELEVEN_DADOS'] = '/content/drive/MyDrive/ElevenAudio/dados'

# Inicia o Eleven Audio Studio (porta 7800) em segundo plano
print("Inicializando interface Eleven Audio (Porta 7800)...")
p_app = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'app.servidor:app', '--host', '127.0.0.1', '--port', '7800'],
    cwd='/content/eleven-audio',
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

# Inicia o túnel Cloudflare para a porta 7800
print("Abrindo túnel Cloudflare seguro...")
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7800'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in tunnel_proc.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n" + "="*70)
    print(f"🎉 ELEVEN AUDIO NO AR COM SUCESSO!")
    print(f"👉 ACESSE SEU ESTÚDIO AQUI: {public_url}")
    print("="*70 + "\n")
else:
    print("⚠️ Não foi possível obter a URL do Cloudflare Tunnel automaticamente.")
    print("Você também pode usar: http://localhost:7800 se estiver usando túnel local.")

# Mantém o notebook rodando
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Encerrando serviços...")
    p_base.terminate()
    p_app.terminate()
    tunnel_proc.terminate()